# 16 — SCD2 Pattern

Purpose: keep historical versions of dimension records.

SCD2 = Slowly Changing Dimension Type 2.

Process schema:

```text
CURRENT DIMENSION
+---------+---------+---------+------------+----------+------------+
| user_id | country | segment | valid_from | valid_to | is_current |
+---------+---------+---------+------------+----------+------------+
| u1      | SK      | retail  | 2026-01-01 | null     | true       |
| u2      | CZ      | retail  | 2026-01-01 | null     | true       |
+---------+---------+---------+------------+----------+------------+

INCOMING CHANGES
+---------+---------+---------+------------+
| user_id | country | segment | change_date|
+---------+---------+---------+------------+
| u1      | SK      | premium | 2026-02-01 | changed
| u3      | AT      | retail  | 2026-02-01 | new
+---------+---------+---------+------------+

SCD2 RESULT
- expire old current row for changed keys
- insert new current row
- insert brand new keys
```

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("scd2-pattern")
    .master("spark://spark-master:7077")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.warehouse.dir", "/tmp/spark_patterns_warehouse")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.version

'4.0.2'

## Step 1 — Current dimension

In [2]:
current_schema = StructType([
    StructField("user_id", StringType(), False),
    StructField("country", StringType(), False),
    StructField("segment", StringType(), False),
    StructField("valid_from_raw", StringType(), False),
    StructField("valid_to_raw", StringType(), True),
    StructField("is_current", BooleanType(), False),
])

current_rows = [
    ("u1", "SK", "retail", "2026-01-01", None, True),
    ("u2", "CZ", "retail", "2026-01-01", None, True),
    ("u4", "DE", "premium", "2026-01-01", None, True),
]

dim_current = (
    spark.createDataFrame(current_rows, current_schema)
    .withColumn("valid_from", F.to_date("valid_from_raw"))
    .withColumn("valid_to", F.to_date("valid_to_raw"))
    .drop("valid_from_raw", "valid_to_raw")
)

dim_current.show(truncate=False)

+-------+-------+-------+----------+----------+--------+
|user_id|country|segment|is_current|valid_from|valid_to|
+-------+-------+-------+----------+----------+--------+
|u1     |SK     |retail |true      |2026-01-01|NULL    |
|u2     |CZ     |retail |true      |2026-01-01|NULL    |
|u4     |DE     |premium|true      |2026-01-01|NULL    |
+-------+-------+-------+----------+----------+--------+



## Step 2 — Incoming changes

In [3]:
incoming_schema = StructType([
    StructField("user_id", StringType(), False),
    StructField("country", StringType(), False),
    StructField("segment", StringType(), False),
    StructField("change_date_raw", StringType(), False),
])

incoming_rows = [
    ("u1", "SK", "premium", "2026-02-01"),
    ("u2", "CZ", "retail", "2026-02-01"),
    ("u3", "AT", "retail", "2026-02-01"),
]

incoming = (
    spark.createDataFrame(incoming_rows, incoming_schema)
    .withColumn("change_date", F.to_date("change_date_raw"))
    .drop("change_date_raw")
)

incoming.show(truncate=False)

+-------+-------+-------+-----------+
|user_id|country|segment|change_date|
+-------+-------+-------+-----------+
|u1     |SK     |premium|2026-02-01 |
|u2     |CZ     |retail |2026-02-01 |
|u3     |AT     |retail |2026-02-01 |
+-------+-------+-------+-----------+



## Step 3 — Detect new, changed, unchanged

Business comparison columns:

```text
country
segment
```

In [4]:
current_only = dim_current.filter(F.col("is_current"))

joined = (
    incoming.alias("i")
    .join(current_only.alias("c"), on="user_id", how="left")
    .select(
        F.col("i.user_id"),
        F.col("i.country").alias("new_country"),
        F.col("i.segment").alias("new_segment"),
        F.col("i.change_date"),
        F.col("c.country").alias("old_country"),
        F.col("c.segment").alias("old_segment"),
        F.col("c.valid_from").alias("old_valid_from"),
        F.col("c.valid_to").alias("old_valid_to"),
        F.col("c.is_current").alias("old_is_current"),
    )
    .withColumn("is_new_key", F.col("old_country").isNull())
    .withColumn(
        "is_changed",
        (~F.col("is_new_key")) &
        (
            (F.col("new_country") != F.col("old_country")) |
            (F.col("new_segment") != F.col("old_segment"))
        )
    )
    .withColumn(
        "is_unchanged",
        (~F.col("is_new_key")) & (~F.col("is_changed"))
    )
)

joined.show(truncate=False)

+-------+-----------+-----------+-----------+-----------+-----------+--------------+------------+--------------+----------+----------+------------+
|user_id|new_country|new_segment|change_date|old_country|old_segment|old_valid_from|old_valid_to|old_is_current|is_new_key|is_changed|is_unchanged|
+-------+-----------+-----------+-----------+-----------+-----------+--------------+------------+--------------+----------+----------+------------+
|u1     |SK         |premium    |2026-02-01 |SK         |retail     |2026-01-01    |NULL        |true          |false     |true      |false       |
|u3     |AT         |retail     |2026-02-01 |NULL       |NULL       |NULL          |NULL        |NULL          |true      |false     |false       |
|u2     |CZ         |retail     |2026-02-01 |CZ         |retail     |2026-01-01    |NULL        |true          |false     |false     |true        |
+-------+-----------+-----------+-----------+-----------+-----------+--------------+------------+--------------+

## Step 4 — Expire changed current rows

For changed keys, close old current record:

```text
valid_to = change_date - 1 day
is_current = false
```

In [5]:
expired_rows = (
    joined
    .filter(F.col("is_changed"))
    .select(
        "user_id",
        F.col("old_country").alias("country"),
        F.col("old_segment").alias("segment"),
        F.col("old_valid_from").alias("valid_from"),
        F.date_sub(F.col("change_date"), 1).alias("valid_to"),
        F.lit(False).alias("is_current"),
    )
)

expired_rows.show(truncate=False)

+-------+-------+-------+----------+----------+----------+
|user_id|country|segment|valid_from|valid_to  |is_current|
+-------+-------+-------+----------+----------+----------+
|u1     |SK     |retail |2026-01-01|2026-01-31|false     |
+-------+-------+-------+----------+----------+----------+



## Step 5 — Insert new current rows

For changed keys and new keys:

```text
valid_from = change_date
valid_to = null
is_current = true
```

In [6]:
new_current_rows = (
    joined
    .filter(F.col("is_changed") | F.col("is_new_key"))
    .select(
        "user_id",
        F.col("new_country").alias("country"),
        F.col("new_segment").alias("segment"),
        F.col("change_date").alias("valid_from"),
        F.lit(None).cast(DateType()).alias("valid_to"),
        F.lit(True).alias("is_current"),
    )
)

new_current_rows.show(truncate=False)

+-------+-------+-------+----------+--------+----------+
|user_id|country|segment|valid_from|valid_to|is_current|
+-------+-------+-------+----------+--------+----------+
|u1     |SK     |premium|2026-02-01|NULL    |true      |
|u3     |AT     |retail |2026-02-01|NULL    |true      |
+-------+-------+-------+----------+--------+----------+



## Step 6 — Keep unaffected existing rows

In [7]:
changed_keys = joined.filter(F.col("is_changed")).select("user_id")

unaffected_existing = (
    dim_current.alias("d")
    .join(changed_keys.alias("k"), on="user_id", how="left_anti")
)

unaffected_existing.show(truncate=False)

+-------+-------+-------+----------+----------+--------+
|user_id|country|segment|is_current|valid_from|valid_to|
+-------+-------+-------+----------+----------+--------+
|u2     |CZ     |retail |true      |2026-01-01|NULL    |
|u4     |DE     |premium|true      |2026-01-01|NULL    |
+-------+-------+-------+----------+----------+--------+



## Step 7 — Build final SCD2 dimension

In [8]:
scd2_final = (
    unaffected_existing
    .unionByName(expired_rows)
    .unionByName(new_current_rows)
    .orderBy("user_id", "valid_from")
)

scd2_final.show(truncate=False)

+-------+-------+-------+----------+----------+----------+
|user_id|country|segment|is_current|valid_from|valid_to  |
+-------+-------+-------+----------+----------+----------+
|u1     |SK     |retail |false     |2026-01-01|2026-01-31|
|u1     |SK     |premium|true      |2026-02-01|NULL      |
|u2     |CZ     |retail |true      |2026-01-01|NULL      |
|u3     |AT     |retail |true      |2026-02-01|NULL      |
|u4     |DE     |premium|true      |2026-01-01|NULL      |
+-------+-------+-------+----------+----------+----------+



## Step 8 — Control totals

In [9]:
control_totals_schema = StructType([
    StructField("metric", StringType(), False),
    StructField("value", LongType(), False),
])

control_totals = spark.createDataFrame(
    [
        ("current_rows_before", dim_current.count()),
        ("incoming_rows", incoming.count()),
        ("new_keys", joined.filter("is_new_key").count()),
        ("changed_keys", joined.filter("is_changed").count()),
        ("unchanged_keys", joined.filter("is_unchanged").count()),
        ("final_rows", scd2_final.count()),
    ],
    control_totals_schema
)

control_totals.show(truncate=False)

+-------------------+-----+
|metric             |value|
+-------------------+-----+
|current_rows_before|3    |
|incoming_rows      |3    |
|new_keys           |1    |
|changed_keys       |1    |
|unchanged_keys     |1    |
|final_rows         |5    |
+-------------------+-----+



## Final result

```text
SCD2
  |
  +--> keep unchanged current rows
  +--> expire changed old rows
  +--> insert changed new rows
  +--> insert new keys
```

In [10]:
spark.stop()